In [62]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / 'data'


In [63]:
network_df = pd.read_csv(DATA_DIR / 'new_data.csv')
college_geo_df = pd.read_csv(DATA_DIR / 'College_geo.csv')
private_geo_df = pd.read_csv(DATA_DIR / 'Private_Schools_geo.csv')

college_geo_df['college_id'] = college_geo_df['college_id'].astype(str)

print(network_df.shape)


(1048575, 4)


In [64]:
# Remove leading 'ID' from identifier columns
for col in ['college_id', 'ncessch', 'ppin']:
    if col in network_df.columns:
        network_df[col] = network_df[col].astype('string').str.replace(r'^ID', '', regex=True)

# Ensure college_id is numeric for downstream merges
# if 'college_id' in network_df.columns:
    # network_df['college_id'] = pd.to_numeric(network_df['college_id'], errors='coerce').astype('Int64')

network_df[['college_id', 'ncessch', 'ppin']].head()

,college_id,ncessch,ppin
0,45896401,450201000242,
1,101189,,A1900014
2,46114802,360007705522,
3,200800,390434800040,
4,155089,201113000926,


In [65]:
# Create school_type: public if ncessch has a value, private if ppin has a value
ncessch_val = network_df['ncessch'].astype('string').str.strip() if 'ncessch' in network_df.columns else pd.Series(pd.NA, index=network_df.index)
ppin_val = network_df['ppin'].astype('string').str.strip() if 'ppin' in network_df.columns else pd.Series(pd.NA, index=network_df.index)

network_df['school_type'] = pd.NA
network_df.loc[ncessch_val.notna() & (ncessch_val != ''), 'school_type'] = 'public'
network_df.loc[network_df['school_type'].isna() & ppin_val.notna() & (ppin_val != ''), 'school_type'] = 'private'

network_df['school_type'] = network_df['school_type'].astype('string')
network_df[['ncessch', 'ppin', 'school_type']].head()

,ncessch,ppin,school_type
0,450201000242,,public
1,,A1900014,private
2,360007705522,,public
3,390434800040,,public
4,201113000926,,public


In [66]:
# Combine ncessch and ppin into hs_id (each row has one or the other)
ncessch_val = network_df['ncessch'].astype('string').str.strip() if 'ncessch' in network_df.columns else pd.Series(pd.NA, index=network_df.index)
ppin_val = network_df['ppin'].astype('string').str.strip() if 'ppin' in network_df.columns else pd.Series(pd.NA, index=network_df.index)

network_df['hs_id'] = pd.NA
network_df.loc[ncessch_val.notna() & (ncessch_val != ''), 'hs_id'] = ncessch_val
network_df.loc[network_df['hs_id'].isna() & ppin_val.notna() & (ppin_val != ''), 'hs_id'] = ppin_val
network_df['hs_id'] = network_df['hs_id'].astype('string')

network_df.drop(columns=['ncessch', 'ppin'], inplace=True, errors='ignore')
network_df[['college_id', 'hs_id', 'school_type', 'cycle']].head()

,college_id,hs_id,school_type,cycle
0,45896401,450201000242,public,2022
1,101189,A1900014,private,2023
2,46114802,360007705522,public,2020
3,200800,390434800040,public,2022
4,155089,201113000926,public,2020


In [67]:
college_geo_df.drop(columns=['col_pop', 'col_enroll'], inplace=True)
college_geo_df.rename(columns={
    'cname_2': 'col_name'
}, inplace=True)

private_geo_df.drop(columns=['hs_pop', 'hs_enroll'], inplace=True)


In [68]:
network_df["college_id"] = network_df["college_id"].astype(str)

college_merged = network_df.merge(
    college_geo_df,
    on='college_id',
    how='left'
)

priv_col_merged = college_merged.merge(
    private_geo_df,
    on='hs_id',
    how='left'
)
priv_col_merged.head()



,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,...,col_shplength,hs_city,hs_state,hs_zip,hs_ctyname,hs_cty_fips,hs_lat,hs_long,hs_x,hs_y
0,45896401,2022,public,450201000242,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,CHARLESTON,...,299.311171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,101189,2023,private,A1900014,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,MONTGOMERY,...,3348.442980,MONTGOMERY,AL,36106.0,MONTGOMERY,10101.0,32.35395,-86.231971,-86.231971,32.35395
2,46114802,2020,public,360007705522,NEW YORK FILM ACADEMY - NEW YORK CAMPUS,NEW YORK,NY,10004.0,3.0,NEW YORK,...,400.965022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,200800,2022,public,390434800040,UNIVERSITY OF AKRON MAIN CAMPUS,AKRON,OH,44325.0,1.0,SUMMIT,...,7848.122407,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,155089,2020,public,201113000926,FRIENDS UNIVERSITY,WICHITA,KS,67213.0,2.0,SEDGWICK,...,3099.852284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
import requests

url = "https://educationdata.urban.org/api/v1/schools/ccd/directory/2023/"

all_rows = []
while url:
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()

    all_rows.extend(data["results"])
    url = data["next"]   # will be None when there are no more pages

pub_df = pd.DataFrame(all_rows)

pub_df.shape
pub_df.head()


,year,ncessch,school_id,school_name,leaid,lea_name,state_leaid,seasch,street_mailing,city_mailing,...,high_cedp,middle_cedp,ungrade_cedp,enrollment,state_leg_district_lower,state_leg_district_upper,ncessch_num,congress_district_id,direct_certification,lunch_program
0,2023,010000500870,0100870,Albertville Middle School,0100005,Albertville City,AL-101,101-0010,600 E Alabama Ave,Albertville,...,0,1,0,884.0,01026,01009,10000500870,104,654.0,2.0
1,2023,010000500871,0100871,Albertville High School,0100005,Albertville City,AL-101,101-0020,402 E McCord Ave,Albertville,...,1,0,0,1710.0,01026,01009,10000500871,104,1225.0,2.0
2,2023,010000500879,0100879,Albertville Intermediate School,0100005,Albertville City,AL-101,101-0110,901 W McKinney Ave,Albertville,...,0,0,0,838.0,01026,01009,10000500879,104,640.0,2.0
3,2023,010000500889,0100889,Albertville Elementary School,0100005,Albertville City,AL-101,101-0200,145 West End Drive,Albertville,...,0,0,0,929.0,01026,01009,10000500889,104,708.0,2.0
4,2023,010000501616,0101616,Albertville Kindergarten and PreK,0100005,Albertville City,AL-101,101-0035,257 Country Club Rd,Albertville,...,0,0,0,598.0,01026,01009,10000501616,104,330.0,2.0


In [34]:
print(pub_df.columns)

Index(['year', 'ncessch', 'school_id', 'school_name', 'leaid', 'lea_name',
       'state_leaid', 'seasch', 'street_mailing', 'city_mailing',
       'state_mailing', 'zip_mailing', 'street_location', 'city_location',
       'state_location', 'zip_location', 'phone', 'fips', 'latitude',
       'longitude', 'csa', 'cbsa', 'urban_centric_locale', 'county_code',
       'school_level', 'school_type', 'school_status', 'lowest_grade_offered',
       'highest_grade_offered', 'bureau_indian_education', 'title_i_status',
       'title_i_eligible', 'title_i_schoolwide', 'charter', 'magnet',
       'shared_time', 'virtual', 'teachers_fte', 'free_lunch',
       'reduced_price_lunch', 'free_or_reduced_price_lunch', 'elem_cedp',
       'high_cedp', 'middle_cedp', 'ungrade_cedp', 'enrollment',
       'state_leg_district_lower', 'state_leg_district_upper', 'ncessch_num',
       'congress_district_id', 'direct_certification', 'lunch_program'],
      dtype='str')


In [51]:
cols_to_keep = ["ncessch", "city_location", "state_location", "zip_location", "county_code", "latitude", "longitude", ]
pub_geo = pub_df[cols_to_keep].copy()

pub_geo.head()

,ncessch,city_location,state_location,zip_location,county_code,latitude,longitude
0,010000500870,Albertville,AL,35950,1095,34.2602,-86.206200
1,010000500871,Albertville,AL,35950,1095,34.2622,-86.204900
2,010000500879,Albertville,AL,35950,1095,34.2733,-86.220100
3,010000500889,Albertville,AL,35950,1095,34.2527,-86.221806
4,010000501616,Albertville,AL,35951,1095,34.2898,-86.193300


In [69]:
priv_col_merged['hs_id'] = priv_col_merged['hs_id'].astype(str)
pub_geo['ncessch'] = pub_geo['ncessch'].astype(str)

priv_col_merged = priv_col_merged.merge(
    pub_geo,
    left_on='hs_id',
    right_on='ncessch',
    how='left',
    suffixes=('', '_pub')
)

# Drop merge key from the public-school table
priv_col_merged.drop(columns=['ncessch'], inplace=True, errors='ignore')

# Fill in empty hs columns with public school data
priv_col_merged['hs_city'] = priv_col_merged['hs_city'].fillna(priv_col_merged['city_location'])
priv_col_merged['hs_state'] = priv_col_merged['hs_state'].fillna(priv_col_merged['state_location'])
priv_col_merged['hs_zip'] = priv_col_merged['hs_zip'].fillna(priv_col_merged['zip_location'])
priv_col_merged['hs_cty_fips'] = priv_col_merged['hs_cty_fips'].fillna(priv_col_merged['county_code'])
priv_col_merged['hs_lat'] = priv_col_merged['hs_lat'].fillna(priv_col_merged['latitude'])
priv_col_merged['hs_long'] = priv_col_merged['hs_long'].fillna(priv_col_merged['longitude'])

# Drop the extra pub columns
priv_col_merged.drop(columns=['city_location', 'state_location', 'zip_location', 'latitude', 'longitude', 'county_code'], inplace=True)

priv_col_merged.head()

,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,...,col_shplength,hs_city,hs_state,hs_zip,hs_ctyname,hs_cty_fips,hs_lat,hs_long,hs_x,hs_y
0,45896401,2022,public,450201000242,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418.0,3.0,CHARLESTON,...,299.311171,Summerville,SC,29483,NaN,45035,32.99010,-80.216592,NaN,NaN
1,101189,2023,private,A1900014,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109.0,2.0,MONTGOMERY,...,3348.442980,MONTGOMERY,AL,36106.0,MONTGOMERY,10101.0,32.35395,-86.231971,-86.231971,32.35395
2,46114802,2020,public,360007705522,NEW YORK FILM ACADEMY - NEW YORK CAMPUS,NEW YORK,NY,10004.0,3.0,NEW YORK,...,400.965022,NEW YORK,NY,10003,NaN,36061,40.72980,-73.992400,NaN,NaN
3,200800,2022,public,390434800040,UNIVERSITY OF AKRON MAIN CAMPUS,AKRON,OH,44325.0,1.0,SUMMIT,...,7848.122407,Akron,OH,44311,NaN,39153,41.06680,-81.514300,NaN,NaN
4,155089,2020,public,201113000926,FRIENDS UNIVERSITY,WICHITA,KS,67213.0,2.0,SEDGWICK,...,3099.852284,Riverton,KS,66770,NaN,20021,37.07300,-94.705700,NaN,NaN


In [70]:
priv_col_merged.drop(columns=["hs_x", "hs_y"], inplace=True)
print(priv_col_merged.columns)

Index(['college_id', 'cycle', 'school_type', 'hs_id', 'col_name', 'col_city',
       'col_st', 'col_zip', 'col_type', 'col_ctyname', 'col_ctyfips',
       'col_shparea', 'col_shplength', 'hs_city', 'hs_state', 'hs_zip',
       'hs_ctyname', 'hs_cty_fips', 'hs_lat', 'hs_long'],
      dtype='str')


In [ ]:
# Convert columns with NaN to nullable integer type
int_cols = ['col_zip', 'col_type']
for col in int_cols:
    priv_col_merged[col] = priv_col_merged[col].astype('Int64')

print(priv_col_merged.shape)
priv_col_merged.to_csv(DATA_DIR / 'merged_geo.csv', index=False)

(1048575, 20)


In [75]:
# Force these to integers (nullable Int64 so bad values become <NA>)
to_int_cols = ["col_ctyfips", "hs_zip", "hs_cty_fips"]

for c in to_int_cols:
    if c in priv_col_merged.columns:
        s = priv_col_merged[c].astype("string").str.strip()
        s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NA": pd.NA, "N/A": pd.NA})
        # handle things like "12345.0"
        s = s.str.replace(r"\.0$", "", regex=True)
        priv_col_merged[c] = pd.to_numeric(s, errors="coerce").astype("Int64")

priv_col_merged[to_int_cols].dtypes


col_ctyfips    Int64
hs_zip         Int64
hs_cty_fips    Int64
dtype: object

In [77]:
priv_col_merged.dtypes
priv_col_merged.to_csv(DATA_DIR / 'merged_geo.csv', index=False)


In [78]:
# Pull college directory data (inst_control, inst_size) - single year is fine for these
url = "https://educationdata.urban.org/api/v1/college-university/ipeds/directory/2023/"
all_rows = []
while url:
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    all_rows.extend(data["results"])
    url = data["next"]

college_dir_df = pd.DataFrame(all_rows)
college_char = college_dir_df[['unitid', 'inst_control', 'inst_size']].copy()
college_char.rename(columns={'unitid': 'college_id'}, inplace=True)

print("college_char shape:", college_char.shape)
college_char.head()

college_char shape: (6163, 3)


,college_id,inst_control,inst_size
0,100654,1,3
1,100663,1,5
2,100690,2,1
3,100706,1,3
4,100724,1,2


In [79]:
# Pull admissions data for years 2019-2023
years = [2019, 2020, 2021, 2022, 2023]
all_admissions = []

for year in years:
    url = f"https://educationdata.urban.org/api/v1/college-university/ipeds/admissions-enrollment/{year}/"
    while url:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        all_admissions.extend(data["results"])
        url = data["next"]
    print(f"Finished year {year}")

admissions_df = pd.DataFrame(all_admissions)
admissions_df["sex"] = pd.to_numeric(admissions_df["sex"], errors="coerce")
admissions_df = admissions_df[admissions_df["sex"] == 99].copy()
college_admissions = admissions_df[['unitid', 'year', 'number_applied', 'number_admitted', 'number_enrolled_total']].copy()
college_admissions.rename(columns={'unitid': 'college_id', 'year': 'cycle'}, inplace=True)

print("college_admissions shape:", college_admissions.shape)
college_admissions.head()

Finished year 2019
Finished year 2020
Finished year 2021
Finished year 2022
Finished year 2023
college_admissions shape: (7972, 5)


,college_id,cycle,number_applied,number_admitted,number_enrolled_total
2,100654,2019,9579.0,8789.0,1710.0
5,100663,2019,8298.0,6112.0,2346.0
8,100706,2019,5295.0,4372.0,1497.0
11,100724,2019,6674.0,6467.0,1023.0
14,100751,2019,38505.0,31835.0,6764.0


In [80]:
# Calculate acceptance rate and enrollment rate
college_admissions['acceptance_rate'] = college_admissions['number_admitted'] / college_admissions['number_applied']
college_admissions['enrollment_rate'] = college_admissions['number_enrolled_total'] / college_admissions['number_admitted']

college_admissions.head()

,college_id,cycle,number_applied,number_admitted,number_enrolled_total,acceptance_rate,enrollment_rate
2,100654,2019,9579.0,8789.0,1710.0,0.917528,0.194561
5,100663,2019,8298.0,6112.0,2346.0,0.736563,0.383835
8,100706,2019,5295.0,4372.0,1497.0,0.825685,0.342406
11,100724,2019,6674.0,6467.0,1023.0,0.968984,0.158188
14,100751,2019,38505.0,31835.0,6764.0,0.826776,0.212471


In [81]:
# Rename columns to have col_ prefix
college_admissions.rename(columns={
    'number_applied': 'col_number_applied',
    'number_admitted': 'col_number_admitted',
    'number_enrolled_total': 'col_number_enrolled_total',
    'acceptance_rate': 'col_acceptance_rate',
    'enrollment_rate': 'col_enrollment_rate'
}, inplace=True)

college_char.rename(columns={
    'inst_control': 'col_inst_control',
    'inst_size': 'col_inst_size'
}, inplace=True)

print(college_admissions.columns.tolist())
print(college_char.columns.tolist())

['college_id', 'cycle', 'col_number_applied', 'col_number_admitted', 'col_number_enrolled_total', 'col_acceptance_rate', 'col_enrollment_rate']
['college_id', 'col_inst_control', 'col_inst_size']


In [83]:
college_admissions['college_id'] = college_admissions['college_id'].astype(str)
college_char['college_id'] = college_char['college_id'].astype(str)


# Merge college characteristics (on college_id only)
merged_df = priv_col_merged.merge(college_char, on='college_id', how='left')
# Merge admissions data (on college_id and cycle)

merged_df_admissions = merged_df.merge(college_admissions, on=['college_id', 'cycle'], how='left')
print(merged_df_admissions.shape)
# print("merged_df shape:", merged_df.shape)
# merged_df.head()

merged_df.to_csv(DATA_DIR / 'merged_geo.csv', index=False)

(1048575, 27)


In [85]:
# Pull NACUBO endowment data (2019-2022), merge into merged_df, and save
import requests
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Retry/backoff helps when the API (or network) drops connections mid-pagination
session = requests.Session()
retries = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=0.75,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=10, pool_maxsize=10)
session.mount("https://", adapter)
session.headers.update({"User-Agent": "c2i-nacubo-endowments/1.0"})

years = [2019, 2020, 2021, 2022]
all_endowments = []

for year in years:
    url = f"https://educationdata.urban.org/api/v1/college-university/nacubo/endowments/{year}/"
    while url:
        resp = session.get(url, timeout=(15, 120))
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])
        for r in results:
            r.setdefault("year", year)
        all_endowments.extend(results)
        url = data.get("next")
        time.sleep(0.05)
    print(f"Finished NACUBO endowments year {year}")

endow_df = pd.DataFrame(all_endowments)
endow_df = endow_df.reindex(columns=['year', 'unitid', 'endow_total', 'endow_per_fte']).copy()
endow_df.rename(columns={'unitid': 'college_id', 'year': 'cycle'}, inplace=True)

# Coerce types and prevent many-to-many merges
endow_df['college_id'] = pd.to_numeric(endow_df['college_id'], errors='coerce')
endow_df['cycle'] = pd.to_numeric(endow_df['cycle'], errors='coerce')
endow_df['endow_total'] = pd.to_numeric(endow_df['endow_total'], errors='coerce')
endow_df['endow_per_fte'] = pd.to_numeric(endow_df['endow_per_fte'], errors='coerce')
endow_df = endow_df.dropna(subset=['college_id', 'cycle'])
endow_df['college_id'] = endow_df['college_id'].astype(int)
endow_df['cycle'] = endow_df['cycle'].astype(int)
endow_df = endow_df.drop_duplicates(subset=['college_id', 'cycle'])

endow_df.head()

df = endow_df[endow_df['cycle']==2020].copy()
print(df.duplicated(['college_id']).sum())
print(df.shape)

endow_df['college_id'] = endow_df['college_id'].astype(str)

merged_df = merged_df.merge(endow_df, on=['college_id', 'cycle'], how='left')
merged_df.to_csv(DATA_DIR / 'merged_geo.csv', index=False)
print(merged_df.shape)
# merged_df[['college_id', 'cycle', 'endow_total', 'endow_per_fte']].head()

Finished NACUBO endowments year 2019
Finished NACUBO endowments year 2020
Finished NACUBO endowments year 2021
Finished NACUBO endowments year 2022
0
(701, 4)
(1048575, 24)


In [86]:
# Pull school-level enrollment by race (2019-2023)
# Using grade-99 (total) and filtering by each race code
import time

years = [2019, 2020, 2021, 2022, 2023]
race_codes = [1, 2, 3, 4, 5, 6, 7]  # White, Black, Hispanic, Asian, AIAN, NHPI, Two or more
all_school_enroll = []

for year in years:
    for race in race_codes:
        url = f"https://educationdata.urban.org/api/v1/schools/ccd/enrollment/{year}/grade-99/race/?race={race}&sex=99"
        while url:
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                data = resp.json()
                all_school_enroll.extend(data["results"])
                url = data["next"]
            except Exception as e:
                print(f"Error for year {year}, race {race}: {e}")
                url = None
    print(f"Finished year {year}, total rows: {len(all_school_enroll)}")

school_enroll_df = pd.DataFrame(all_school_enroll)
print("school_enroll_df shape:", school_enroll_df.shape)
school_enroll_df.head()

Finished year 2019, total rows: 695030
Finished year 2020, total rows: 1390200
Finished year 2021, total rows: 2090403
Finished year 2022, total rows: 2789423
Finished year 2023, total rows: 3487169
school_enroll_df shape: (3487169, 9)


,year,ncessch,ncessch_num,grade,race,sex,enrollment,fips,leaid
0,2019,010000500870,10000500870,99,1,99,351,1,100005
1,2019,010000500871,10000500871,99,1,99,711,1,100005
2,2019,010000500879,10000500879,99,1,99,378,1,100005
3,2019,010000500889,10000500889,99,1,99,352,1,100005
4,2019,010000501616,10000501616,99,1,99,230,1,100005


In [ ]:
# Pivot school enrollment data to get race percentages
school_pivot = school_enroll_df.pivot_table(
    index=['ncessch', 'year'],
    columns='race',
    values='enrollment',
    aggfunc='sum'
).reset_index()

# Calculate total enrollment and percentages
# Race codes: 1=White, 2=Black, 3=Hispanic, 4=Asian, 5=AIAN, 6=NHPI, 7=Two or more, 9=Unknown, 99=Total
race_cols = [c for c in [1, 2, 3, 4, 5, 6, 7] if c in school_pivot.columns]
school_pivot['total'] = school_pivot[race_cols].sum(axis=1, skipna=True)

school_pivot['hs_pct_white'] = school_pivot.get(1, 0) / school_pivot['total']
school_pivot['hs_pct_black'] = school_pivot.get(2, 0) / school_pivot['total']
school_pivot['hs_pct_hispanic'] = school_pivot.get(3, 0) / school_pivot['total']
school_pivot['hs_pct_asian'] = school_pivot.get(4, 0) / school_pivot['total']
school_pivot['hs_pct_aian'] = school_pivot.get(5, 0) / school_pivot['total']
school_pivot['hs_pct_nhpi'] = school_pivot.get(6, 0) / school_pivot['total']
school_pivot['hs_pct_two_or_more'] = school_pivot.get(7, 0) / school_pivot['total']

# Keep only needed columns
school_race = school_pivot[['ncessch', 'year', 'hs_pct_white', 'hs_pct_black', 'hs_pct_hispanic', 
                            'hs_pct_asian', 'hs_pct_aian', 'hs_pct_nhpi', 'hs_pct_two_or_more']].copy()
school_race.rename(columns={'year': 'cycle'}, inplace=True)

print("school_race shape:", school_race.shape)
school_race.head()

school_race shape: (498167, 10)


race,ncessch,cycle,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more,hs_is_district
0,010000500870,2019,0.407666,0.032520,0.528455,0.003484,0.001161,0.000000,0.026713,0
1,010000500870,2020,0.408590,0.036344,0.516520,0.004405,0.002203,0.000000,0.031938,0
2,010000500870,2021,0.383696,0.039130,0.528261,0.005435,0.004348,0.001087,0.038043,0
3,010000500870,2022,0.353933,0.032584,0.564045,0.006742,0.005618,0.001124,0.035955,0
4,010000500870,2023,0.321267,0.041855,0.599548,0.006787,0.003394,0.000000,0.027149,0


In [88]:
# Merge school race data to main dataframe on hs_id (public) and cycle
school_race['ncessch'] = school_race['ncessch'].astype(str)
merged_df['hs_id'] = merged_df['hs_id'].astype(str)

merged_df = merged_df.merge(
    school_race,
    left_on=['hs_id', 'cycle'],
    right_on=['ncessch', 'cycle'],
    how='left'
)

# Drop merge key from the school_race table
merged_df.drop(columns=['ncessch'], inplace=True, errors='ignore')

print("merged_df shape:", merged_df.shape)
print("Non-null hs_pct_white:", merged_df['hs_pct_white'].notna().sum())
merged_df.head()

merged_df shape: (1048575, 32)
Non-null hs_pct_white: 759780


,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,...,endow_total,endow_per_fte,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more,hs_is_district
0,45896401,2022,public,450201000242,STRAYER UNIVERSITY - CHARLESTON CAMPUS,NORTH CHARLESTON,SC,29418,3,CHARLESTON,...,NaN,NaN,0.503529,0.312941,0.085882,0.008235,0.000000,0.001176,0.088235,0.0
1,101189,2023,private,A1900014,FAULKNER UNIVERSITY,MONTGOMERY,AL,36109,2,MONTGOMERY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,46114802,2020,public,360007705522,NEW YORK FILM ACADEMY - NEW YORK CAMPUS,NEW YORK,NY,10004,3,NEW YORK,...,NaN,NaN,0.086207,0.362069,0.534483,0.017241,0.000000,0.000000,0.000000,0.0
3,200800,2022,public,390434800040,UNIVERSITY OF AKRON MAIN CAMPUS,AKRON,OH,44325,1,SUMMIT,...,NaN,NaN,0.142857,0.574124,0.061995,0.129380,0.000000,0.005391,0.086253,0.0
4,155089,2020,public,201113000926,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,...,62614501.0,54732.95542,0.774194,0.004608,0.018433,0.013825,0.110599,0.004608,0.073733,0.0


In [89]:
merged_df.dtypes

college_id                str
cycle                   int64
school_type            string
hs_id                     str
col_name                  str
col_city                  str
col_st                    str
col_zip                 Int64
col_type                Int64
col_ctyname               str
col_ctyfips             Int64
col_shparea           float64
col_shplength         float64
hs_city                   str
hs_state                  str
hs_zip                  Int64
hs_ctyname                str
hs_cty_fips             Int64
hs_lat                float64
hs_long               float64
col_inst_control      float64
col_inst_size         float64
endow_total           float64
endow_per_fte         float64
hs_pct_white          float64
hs_pct_black          float64
hs_pct_hispanic       float64
hs_pct_asian          float64
hs_pct_aian           float64
hs_pct_nhpi           float64
hs_pct_two_or_more    float64
hs_is_district        float64
dtype: object

In [91]:
merged_df["col_inst_control"] = merged_df["col_inst_control"].astype('Int64')
merged_df["col_inst_size"] = merged_df["col_inst_size"].astype('Int64')

merged_df = merged_df.rename(columns={"endow_total": "col_endow_total", "endow_per_fte": "col_endow_per_fte"})

merged_df.dtypes


college_id                str
cycle                   int64
school_type            string
hs_id                     str
col_name                  str
col_city                  str
col_st                    str
col_zip                 Int64
col_type                Int64
col_ctyname               str
col_ctyfips             Int64
col_shparea           float64
col_shplength         float64
hs_city                   str
hs_state                  str
hs_zip                  Int64
hs_ctyname                str
hs_cty_fips             Int64
hs_lat                float64
hs_long               float64
col_inst_control        Int64
col_inst_size           Int64
col_endow_total       float64
col_endow_per_fte     float64
hs_pct_white          float64
hs_pct_black          float64
hs_pct_hispanic       float64
hs_pct_asian          float64
hs_pct_aian           float64
hs_pct_nhpi           float64
hs_pct_two_or_more    float64
hs_is_district        float64
dtype: object

In [92]:

merged_df = merged_df.drop(columns=["hs_is_district"])


In [93]:
merged_df.to_csv(DATA_DIR / 'merged_geo.csv', index=False)

In [96]:

col_ex = merged_df[merged_df["college_id"] == "155089"]

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don’t wrap based on terminal width
pd.set_option("display.max_colwidth", None)  # don’t truncate long text cells

col_ex.head()


,college_id,cycle,school_type,hs_id,col_name,col_city,col_st,col_zip,col_type,col_ctyname,col_ctyfips,col_shparea,col_shplength,hs_city,hs_state,hs_zip,hs_ctyname,hs_cty_fips,hs_lat,hs_long,col_inst_control,col_inst_size,col_endow_total,col_endow_per_fte,hs_pct_white,hs_pct_black,hs_pct_hispanic,hs_pct_asian,hs_pct_aian,hs_pct_nhpi,hs_pct_two_or_more
4,155089,2020,public,201113000926,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,20173,398472.0508,3099.852284,Riverton,KS,66770,NaN,20021,37.073000,-94.705700,2,2,62614501.0,54732.955420,0.774194,0.004608,0.018433,0.013825,0.110599,0.004608,0.073733
2485,155089,2022,public,201305001211,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,20173,398472.0508,3099.852284,Winfield,KS,67156,NaN,20035,37.235000,-96.995100,2,2,62735229.0,48783.226283,0.661290,0.028226,0.145161,0.028226,0.012097,0.020161,0.104839
10365,155089,2020,public,200336001610,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,20173,398472.0508,3099.852284,Andover,KS,67002,NaN,20015,37.717700,-97.135600,2,2,62614501.0,54732.955420,0.791444,0.018717,0.066845,0.032086,0.010695,0.000000,0.080214
18135,155089,2021,public,483060003458,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,20173,398472.0508,3099.852284,MIDLOTHIAN,TX,76065,NaN,48139,32.472469,-96.994439,2,2,58725123.0,45103.781106,0.562973,0.116477,0.254735,0.010417,0.008996,0.000000,0.046402
22273,155089,2023,private,00489622,FRIENDS UNIVERSITY,WICHITA,KS,67213,2,SEDGWICK,20173,398472.0508,3099.852284,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
